# Installation

In [ ]:
!pip install git+https://github.com/borongzhang/back_projection_diffusion.git@main

In [ ]:
!pip install --upgrade "jax[cuda12_pip]" -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html 

# Imports

In [1]:
from back_projection_diffusion.src import utils, fstars, fstar_cnn

import functools
import os
from clu import metric_writers
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import optax
import orbax.checkpoint as ocp

import h5py
import natsort
import tensorflow as tf
from scipy.ndimage import geometric_transform
from scipy.ndimage import gaussian_filter

from swirl_dynamics import templates
from swirl_dynamics.lib import diffusion as dfn_lib
from swirl_dynamics.lib import solvers as solver_lib
from swirl_dynamics.projects import probabilistic_diffusion as dfn


I0000 00:00:1778185389.888261 1415374 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1778185393.539843 1415374 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
# To avoid tf to use GPU memory
tf.config.set_visible_devices([], device_type='GPU')


W0000 00:00:1778185397.285881 1415374 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


# Dataset

In [3]:
# Parameters for the computational task.

L = 4 # number of levels (even number)
s = 5 # leaf size
r = 3 # rank

# Discretization of Omega (n_eta * n_eta).
neta = (2**L)*s

# Number of sources/detectors (n_sc).
# Discretization of the domain of alpha in polar coordinates (n_theta * n_rho).
# For simplicity, these values are set equal (n_sc = n_theta = n_rho), facilitating computation.
nx = (2**L)*s

# Standard deviation for the Gaussian blur.
blur_sigma = 0.5

# Number of training datapoints.
ntrain = 21000
batch_size_train = 16

# Number of testing datapoints.
ntest = 500
batch_size_test = 10


In [4]:
training_eta_path = os.path.abspath('..') + '/data/10hsquares_trainingdata/eta.h5'
training_scatter_path = os.path.abspath('..') + '/data/10hsquares_trainingdata/scatter.h5'
eta_train, mean_eta, std_eta = utils.load_eta_data(training_eta_path, ntrain, blur_sigma=0.5, normalize=True)
scatter_train, norm_constants = utils.load_scatter_data(training_scatter_path, ntrain, scatter_norm_constants=None)

test_eta_path = os.path.abspath('..') + '/data/10hsquares_testdata/eta.h5'
test_scatter_path = os.path.abspath('..') + '/data/10hsquares_testdata/scatter_order_8.h5'
eta_test = utils.load_eta_data(test_eta_path, ntest, blur_sigma=0.5, normalize=False)
scatter_test = utils.load_scatter_data(test_scatter_path, ntest, scatter_norm_constants=norm_constants)


In [5]:
eta_train = eta_train.reshape(-1, 80, 80, 1)
scatter_train = scatter_train.reshape(-1, 6400, 2, 3) 
dataset = utils.create_dataset(eta_train, scatter_train, batch_size=batch_size_train, repeat=True)

eta_test = eta_test.reshape(-1, 80, 80, 1)
scatter_test = scatter_test.reshape(-1, 6400, 2, 3)
c = 0.0 # percentage of noise to add
scatter_test += np.random.normal(0, c, size = scatter_test.shape)
dataset_test = utils.create_dataset(eta_test, scatter_test, batch_size=batch_size_test, repeat=False)


# Architecture

In [6]:
r_index = utils.rotation_index(nx)
cart_mat = utils.sparse_polar_to_cartesian(neta, nx)


Processing row 80 / 80

In [7]:
# a list of NN approximations of back-scattering operator for each frequency 
# n_freq can be changed based on how many frequencies the data has.
n_freq = 3

fstarlist = [fstars.EquinetFstar( 
    nx = nx, 
    neta = neta,
    cart_mat = cart_mat,
    r_index = r_index
) for i in range(n_freq)]


In [8]:
cond_denoiser_model = fstar_cnn.PreconditionedDenoiser(
    fstars=fstarlist,
    out_channels=1,
    squeeze_ratio=8,
    cond_embed_iter=10, 
    noise_embed_dim=96, 
    num_conv=8,
    num_feature=96, # multiples of 32
)


In [9]:
diffusion_scheme = dfn_lib.Diffusion.create_variance_preserving(
    sigma=dfn_lib.tangent_noise_schedule(),
    data_std=1, # we always use normalized data
)

cond_model = dfn.DenoisingModel(
    input_shape=(80,80,1),
    cond_shape={"channel:scatter0": (6400,2),
                "channel:scatter1": (6400,2),
                "channel:scatter2": (6400,2)},
    denoiser=cond_denoiser_model,
    noise_sampling=dfn_lib.time_uniform_sampling(
        diffusion_scheme, clip_min=1e-4, uniform_grid=True,
    ),
    noise_weighting=dfn_lib.edm_weighting(data_std=1),
)


In [10]:
rng = jax.random.PRNGKey(888)
params = cond_model.initialize(rng)
total_parameters = utils.count_params(params)
print(f"Total parameters in the model: {total_parameters}")


Layer: params/fstars_0/pre1, Parameters: 80
Layer: params/fstars_0/pre2, Parameters: 80
Layer: params/fstars_0/pre3, Parameters: 80
Layer: params/fstars_0/pre4, Parameters: 80
Layer: params/fstars_0/post1, Parameters: 80
Layer: params/fstars_0/post2, Parameters: 80
Layer: params/fstars_0/post3, Parameters: 80
Layer: params/fstars_0/post4, Parameters: 80
Layer: params/fstars_0/cos_kernel1, Parameters: 6400
Layer: params/fstars_0/sin_kernel1, Parameters: 6400
Layer: params/fstars_0/cos_kernel2, Parameters: 6400
Layer: params/fstars_0/sin_kernel2, Parameters: 6400
Layer: params/fstars_0/cos_kernel3, Parameters: 6400
Layer: params/fstars_0/sin_kernel3, Parameters: 6400
Layer: params/fstars_0/cos_kernel4, Parameters: 6400
Layer: params/fstars_0/sin_kernel4, Parameters: 6400
Layer: params/fstars_1/pre1, Parameters: 80
Layer: params/fstars_1/pre2, Parameters: 80
Layer: params/fstars_1/pre3, Parameters: 80
Layer: params/fstars_1/pre4, Parameters: 80
Layer: params/fstars_1/post1, Parameters: 80

# Training

In [11]:
epochs = 100
num_train_steps = 21000 * epochs // 16  #@param
cond_workdir = os.path.abspath('') + "/tmp/equinet_cnn_10hsquares"
initial_lr = 1e-5 #@param
peak_lr = 1e-3 #@param
warmup_steps = num_train_steps // 20  #@param
end_lr = 1e-8 #@param
ema_decay = 0.999  #@param
ckpt_interval = 2000 #@param
max_ckpt_to_keep = 3 #@param


In [12]:
cond_trainer = dfn.DenoisingTrainer(
    model=cond_model,
    rng=jax.random.PRNGKey(888),
    optimizer=optax.adam(
        learning_rate=optax.warmup_cosine_decay_schedule(
            init_value=initial_lr,
            peak_value=peak_lr,
            warmup_steps=warmup_steps,
            decay_steps=num_train_steps,
            end_value=end_lr,
        ),
    ),
    ema_decay=ema_decay,
)


In [14]:
templates.run_train(
    train_dataloader=dataset,
    trainer=cond_trainer,
    workdir=cond_workdir,
    total_train_steps=num_train_steps,
    metric_writer=metric_writers.create_default_writer(
        cond_workdir, asynchronous=False
    ),
    metric_aggregation_steps = 100,
    callbacks=(
        templates.TqdmProgressBar(
            total_train_steps=num_train_steps,
            train_monitors=("train_loss",),
        ),
        templates.TrainStateCheckpoint(
            base_dir=cond_workdir,
            options=ocp.CheckpointManagerOptions( 
                save_interval_steps=ckpt_interval, max_to_keep=max_ckpt_to_keep
            ),
        ),
    ),
)


  0%|          | 0/131250 [00:00<?, ?step/s]

# Inference

In [15]:
trained_state = dfn.DenoisingModelTrainState.restore_from_orbax_ckpt(
    f"{cond_workdir}/checkpoints", step=None
)

# Construct the inference function
cond_denoise_fn = dfn.DenoisingTrainer.inference_fn_from_state_dict(
    trained_state, use_ema=True, denoiser=cond_denoiser_model
)


In [16]:
cond_sampler = dfn_lib.SdeSampler(
    input_shape=(80,80,1),
    integrator=solver_lib.EulerMaruyama(),
    tspan=dfn_lib.exponential_noise_decay(diffusion_scheme, num_steps=256, end_sigma=1e-3),
    scheme=diffusion_scheme,
    denoise_fn=cond_denoise_fn,
    guidance_transforms=(),
    apply_denoise_at_end=True,
    return_full_paths=False,
)


We JIT the generate function for the sake of faster repeated sampling calls. Here we employ `functools.partial` to specify `num_samples_per_cond`, making it easier to vectorize across the batch dimension with `jax.vmap`.

In [17]:
num_samples_per_cond = 1 # Choose the number of samples for each condition

generate = jax.jit(
    functools.partial(cond_sampler.generate, num_samples_per_cond)
)

In [20]:
eta_pred = np.zeros((ntest, num_samples_per_cond, neta, neta, 1))

b = 0
for batch in dataset_test:
    print(f"\rProcessing batch {b + 1} / {ntest//batch_size_test}", end='', flush=True)
    cond_samples = jax.device_get(jax.vmap(generate, in_axes=(0, 0, None))(
        jax.random.split(jax.random.PRNGKey(68), batch_size_test),
        batch["cond"],
        None,  # Guidance inputs = None since no guidance transforms involved  
    ))
    eta_pred[b*batch_size_test:(b+1)*batch_size_test,:,:,:,:] = cond_samples*std_eta+mean_eta[:, :, jnp.newaxis]
    b += 1
    

Processing batch 50 / 50

In [21]:
errors = []
for i in range(ntest):
    for j in range(num_samples_per_cond):
        errors.append(np.linalg.norm(eta_test[i,:,:,0]-eta_pred[i,0,:,:,0])/np.linalg.norm(eta_test[i,:,:,0]))
        
print('Mean of validation relative l2 error:', np.mean(errors))
print('Median of validation relative l2 error:', np.median(errors))
print('Min of validation relative l2 error:', np.min(errors))
print('Max of validation relative l2 error:', np.max(errors))
print('Standard deviation of validation relative l2 errors:', np.std(errors))


Mean of validation relative l2 error: 0.018612519909338893
Median of validation relative l2 error: 0.01163870658105081
Min of validation relative l2 error: 0.005904624311067795
Max of validation relative l2 error: 0.10564049862453856
Standard deviation of validation relative l2 errors: 0.018186720909439676


In [ ]:
#with h5py.File("results_equinet_cnn_squares.h5", "w") as f:
#    f.create_dataset('eta', data=eta_test)
#    f.create_dataset('eta_pred', data=eta_pred)